## TEST Baseline 2 (OOP Version)

Object 폴더의 모듈을 import하여 객체지향적으로 실험을 진행한다.

| Step | 내용 |
|---|---|
| 1 | 파일 경로 확인 |
| 2 | 데이터 로드 |
| 3 | 데이터 분석 방향 설정 |
| 4 | 데이터 전처리 및 시각화 |
| 5 | 평가지표 계산 및 손실함수 통한 모형 적합 |
| 6 | 그룹별(KPX_1 / 2/ 3) 개별 모형 학습 |
| 7 |  |
| 8 | 스키마 검증 |
| 9 | 제출 CSV 저장 |

In [ ]:
import os
import sys

import csv
from typing import List
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import statsmodels.api as stats

In [ ]:
import torch
import torch.nn as nn
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
# pip install --upgrade typing-extensions --force-reinstall
# pip install --upgrade pip

In [ ]:
ROOT = "/Users/ksydata/WINDFORCE"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
# Windforce 패키지 경로를 sys.path에 추가

from Windforce import (
    WindforceDataLoader,
    FeatureEngineer,
    LDAPSFeatureEngineer,
    GFSFeatureEngineer,
    EvaluationMetrics,
    RATED_CAPACITY_KW,
    TIME_STEP_HOURS,
    ScoreLossFunction,
)

from Windforce.Modeling import (
    LSTMPipeline,
    GroupExperimentRunner,
    BaselineModels
)
# Windforce 패키지에서 클래스 import

In [ ]:
pd.set_option("display.max_columns", None)
# 데이터프레임 생략되는 열 없이 모든 열 조회
pd.options.display.float_format = "{:,.8f}".format
# pd.options.display.float_format = None # 원상복구
# 파이썬 지수 e표현 실수로 변환

## Step 1. 파일 경로 확인
모든 경로 내 파일이 실제로 존재하는지 확인한다.

In [ ]:
loader = WindforceDataLoader()
# 클래스 인스턴스 생성 (root 기본값 사용)
loader.check_paths()
# 파일 존재 여부 먼저 확인

## Step 2. 데이터 로드 및 구조 확인
- 풍력발전량 예측에서 입력으로 봐야 할 기상 변수는 무엇인가?
- 시간 단위 예측값 제출에서 실수하기 쉬운 포맷은 무엇인가?
- NMAE와 정산금획득률 중 어느 지표가 내 실험 로그에 먼저 들어가야 하는가?

In [ ]:
# loader.load_all()
# 전체 로딩 및 shape 출력

In [ ]:
print(loader["scada_vestas_train"].info(), loader["scada_unison_train"].info())
# 풍속(Wind Speed) : 가장 중요한 독립변수, 예측 모델의 핵심 입력
# 풍향(Wind Direction) : 요 편차 계산, 효율 손실 분석

In [ ]:
print(loader["train_labels"].head(5))
print(loader["sample_submission"].head(5))

In [ ]:
loader["kpx_info"]

In [ ]:
loader["ldaps_train"].head(1)
# LDAPS 기반 기상 예보 데이터로 각각 학습 기간과 평가 기간의 기상 예보 정보 포함

In [ ]:
loader["gfs_train"].head(1)
# GFS 기반 기상 예보 데이터로 각각 학습 기간과 평가 기간의 기상 예보 정보 포함

---
## Step 4. 데이터 전처리 및 시각화
1. SCADA 데이터와 Power Curve 비교
https://ihatenumber.tistory.com/entry/%ED%8C%8C%EC%9D%B4%EC%8D%AC-%EC%9C%A0%EC%86%8D-%ED%92%8D%EC%86%8Dspeed-%EC%9C%A0%ED%96%A5-%ED%92%8D%ED%96%85degree%EC%9D%98-U-V-%EB%B3%80%ED%99%98

In [ ]:
# LDAPSFeatureEngineer, GFSFeatureEngineer 인스턴스 생성
ldaps_engineer = LDAPSFeatureEngineer()
gfs_engineer = GFSFeatureEngineer()

# 각 기상 데이터 전처리 (풍속·풍향 파생변수 생성)
ldaps_transformed_df = ldaps_engineer.transformLDAPS(loader["ldaps_train"])
gfs_transformed_df   = gfs_engineer.transformGFS(loader["gfs_train"])

In [ ]:
ldaps_transformed_df.head(2)

In [ ]:
gfs_transformed_df.head(2)

In [ ]:
loader["scada_vestas_train"].head(1)

In [ ]:
# 풍력단지별 터빈 단위 SCADA 풍력, 풍속, 발전량 데이터(VESTAS 제작사, 12개 호기)
for columnName in ["vestas_wtg01", "vestas_wtg02", "vestas_wtg03",
                   "vestas_wtg04", "vestas_wtg05", "vestas_wtg06",
                   "vestas_wtg07", "vestas_wtg08", "vestas_wtg09",
                   "vestas_wtg10", "vestas_wtg11", "vestas_wtg12"]:
    print(columnName, "\n", loader["scada_vestas_train"].filter(like=columnName).describe())
    # 이상치로 추정되는 발전량 유의(마이너스 값 또는 설비용량 600kw을 초과하는 값)

In [ ]:
loader["scada_unison_train"].head(1)

In [ ]:
# 풍력단지별 터빈 단위 SCADA 풍력, 풍속, 발전량 데이터(UNISON 제작사, 5개 호기)
for columnName in ["unison_wtg01", "unison_wtg02", "unison_wtg03",
                   "unison_wtg04", "unison_wtg05"]:
    print(columnName, "\n", loader["scada_unison_train"].filter(like=columnName).describe())

In [ ]:
# 풍력단지별 터빈 단위 SCADA 풍향, 풍속, 발전량 데이터 3D 산점도
target_df = loader["scada_unison_train"]

target_cols = []
for col in target_df.columns:
    if col.startswith("unison_wtg"):
        base_name = col.replace("_ws", "").replace("_wd", "").replace("_power_kw10m", "")
        if base_name not in target_cols:
            target_cols.append(base_name)

print(f"Found {len(target_cols)} turbines for visualization.")

for col in target_cols:
    ws_col = f"{col}_ws"
    wd_col = f"{col}_wd"
    pw_col = f"{col}_power_kw10m"

    if (ws_col in target_df.columns) and (wd_col in target_df.columns) and (pw_col in target_df.columns):
        fig = px.scatter_3d(
            target_df,
            x=ws_col, y=wd_col, z=pw_col,
            color=pw_col,
            color_continuous_scale="Viridis",
            title=f"[{col}] Wind Speed vs Direction vs Power",
            labels={ws_col: "Wind Speed (m/s)", wd_col: "Wind Direction (°)", pw_col: "Power (kW)"},
            opacity=0.7
        )
        fig.update_layout(margin=dict(l=0, r=0, b=0, t=40))
        fig.show()
        break  # 첫 번째 터빈만 시각화

In [ ]:
# 풍속 vs 발전량 2D 산점도
for col in target_cols:
    ws_col = f"{col}_ws"
    pw_col = f"{col}_power_kw10m"

    if (ws_col in target_df.columns) and (pw_col in target_df.columns):
        fig = px.scatter(
            target_df,
            x=ws_col, y=pw_col,
            title=f"[{col}] Wind Speed vs Power",
            labels={ws_col: "Wind Speed (m/s)", pw_col: "Power (kW)"},
            opacity=0.4,
            color=pw_col,
            color_continuous_scale="Cividis"
        )
        fig.update_layout(margin=dict(l=0, r=0, b=0, t=30))
        fig.show()
        break

In [ ]:
# 파워 커브 (LOWESS 트렌드라인)
for col in target_cols:
    ws_col = f"{col}_ws"
    pw_col = f"{col}_power_kw10m"

    if (ws_col in target_df.columns) and (pw_col in target_df.columns):
        curve_df = target_df[[ws_col, pw_col]].dropna()
        curve_df = curve_df[(curve_df[ws_col] > 0.5) & (curve_df[pw_col] >= 0)]
        if len(curve_df) > 3000:
            curve_df = curve_df.sample(3000)

        fig = px.scatter(
            curve_df,
            x=ws_col, y=pw_col,
            trendline="lowess",
            trendline_color_override="red",
            title=f"[{col}] Power Curve (Wind Speed vs Power)",
            labels={ws_col: "Wind Speed (m/s)", pw_col: "Power (kW)"},
            opacity=0.4,
            template="plotly_white"
        )
        fig.update_layout(margin=dict(l=0, r=0, b=0, t=40))
        fig.show()
        break

## Step 5. 평가지표 계산 및 손실함수 통한 모형 적합

1. 평가지표 (대회 규정)
   - 1-NMAE : 그룹별 NMAE_g = mean(|pred-actual|/설비용량) 을 3개 그룹 평균낸 뒤 1에서 뺀 값. 높을수록 좋음.
   - FICR   : 시간별 NMAE 구간에 따라 FICR의 단가(6% 이하: 4원, 6~8%: 3원, 8% 초과: 0원)가 달라짐.
           FICR_g = 획득 정산금 / 이론상 최대 정산금(=항상 NMAE<=6% 가정 시 4원 단가 적용).
           3개 그룹 FICR 평균.
   - 총점   : 0.5*(1-NMAE) + 0.5*FICR
2. 손실함수
   - loss = 1 - 총점 = 0.5*NMAE + 0.5*(1-FICR)
      - NMAE 항: |예측값-실제값| 그대로 사용 (미분 가능)
      - FICR 항: 정산 단가 계단함수(4/3/0원)를 시그모이드로 완화해 미분 가능하게 근사
        k가 클수록 실제 계단함수에 가까워짐 (기본값 300).
      - actual_power_kw: 발전량 가중치(FICR은 발전량 가중 평균이므로 필요).

In [ ]:
# [테스트 코드] Step 5.1. 평가지표(NMAE, FICR) 계산
# EvaluationMetrics 인스턴스 생성 + train_labels 구조 확인

metrics = EvaluationMetrics(
    rated_capacity_kw = RATED_CAPACITY_KW,
    time_step_hours = TIME_STEP_HOURS
)
# EvaluationMetrics 인스턴스 생성

labels_df = loader["train_labels"]
print(labels_df.head(3))
print(labels_df.columns.tolist())
# 더미 예측값으로 평가 흐름 검증

In [ ]:
# [테스트 코드] Step 5.2. 손실함수 계산

loss_fns = {
    g: ScoreLossFunction(capacity_kw=RATED_CAPACITY_KW[g])
    for g in [1, 2, 3]
}
# ScoreLossFunction 인스턴스 생성 (그룹별 설비용량으로 초기화)

dummy_pred   = torch.tensor([100.0, 200.0, 300.0])
dummy_actual = torch.tensor([110.0, 190.0, 310.0])
for g, fn in loss_fns.items():
    loss_val = fn(dummy_pred, dummy_actual)
    print(f"Group {g} loss: {loss_val.item():.6f}")
# 손실함수 동작 확인 (더미 텐서 사용)    

## Step 6. 그룹별(KPX_1 / 2/ 3) 개별 모형 학습

```
피처 엔지니어링(Step 4) → X (모델 입력)
                        ↓
                  모델(LSTM 등, Step 7)
                        ↓
                  pred (kWh)  ──┐
                                ├── EvaluationMetrics(Step 5) / ScoreLossFunction(Step 6)
                  actual (kWh) ─┘
```                          

In [ ]:
GROUPS  = [1, 2, 3]
SEQ_LEN = 24   # 최근 24시간 시퀀스로 다음 시점 예측
SEED    = 42
np.random.seed(SEED)
torch.manual_seed(SEED)


class WindforceDatasetBuilder:
    """LDAPS/GFS 파생 피처와 train_labels를 결합해 그룹별 학습용 데이터프레임을 만드는 클래스"""

    def __init__(self, ldaps_df: pd.DataFrame, gfs_df: pd.DataFrame, labels_df: pd.DataFrame):
        self.ldaps_df  = ldaps_df
        self.gfs_df    = gfs_df
        self.labels_df = labels_df

In [ ]:
# WindforceDatasetBuilder 인스턴스 생성
dataset_builder = WindforceDatasetBuilder(
    ldaps_df  = ldaps_transformed_df,
    gfs_df    = gfs_transformed_df,
    labels_df = loader["train_labels"]
)
print("DatasetBuilder ready:", dataset_builder)

In [ ]:
# 실행: EvaluationMetrics 생성 → GroupExperimentRunner로 전체 실험 실행 → 결과 출력

metrics = EvaluationMetrics(
    rated_capacity_kw=RATED_CAPACITY_KW,
    time_step_hours=TIME_STEP_HOURS
)
# Step5의 평가지표 계산기 생성 (그룹 공용, 상태 없음)

runner = GroupExperimentRunner(datasetBuilder=dataset_builder, metrics=metrics)
# dataset_builder는 이전 셀에서 이미 생성해둔 WindforceDatasetBuilder 인스턴스를 그대로 주입

resultDf = runner.runAll()
# 3개 그룹 x 3개 모델 = 9개 결과가 self.results에 누적되고, DataFrame으로 반환됨

print(resultDf.pivot(index="model_name", columns="group", values=["nmae", "ficr", "score"]))
# 모델(행) x 그룹(열) 형태로 nmae/ficr/score를 한눈에 볼 수 있게 피벗

finalScores = runner.finalScores()
# 모델별로 3개 그룹을 평균 낸 대회 정식 총점 계산

print("\n최종 총점 (3그룹 평균 기준):")
for name, score in finalScores.items():
    print(f"  {name}: {score:.4f}")